# 적대적 AI — GPU 학습 및 복구
Colab GPU 런타임을 선택하고 세 셀을 순서대로 실행하세요. 실제 GPU 통합 실행은 아직 미검증입니다.
비공개 ZIP은 내 Drive 또는 접근 가능한 공유폴더에 둬도 됩니다. 자동 검색 시 Google 인증이 한 번 더 필요할 수 있습니다.
학습은 완료된 epoch의 모델·최적화 상태를 Drive에 저장합니다. 중단 후 같은 RUN과 RESUME=True로 복구하며 미완료 epoch는 다시 계산합니다.
CPU 결과와 분리된 새 GPU 실험입니다. GPU가 없으면 중단하며 유료 리소스를 자동 구매하지 않습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess,sys,os,json,zipfile,shutil,datetime,hashlib
ARCHIVE=Path('/content/drive/MyDrive/AdversarialAI_GPU_handoff_20261002.zip')
WORK=Path('/content/adversarial_mobile');WORK.mkdir(exist_ok=True)
REPO=WORK/'repo'
COMMIT='76fc5e5debfc02269e236d15db5d06325802379a'
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/heechan9/AdversarialAI_Security.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin',COMMIT],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==COMMIT
if not ARCHIVE.is_file():
    from google.colab import auth
    auth.authenticate_user()
    import google.auth
    from googleapiclient.discovery import build
    sys.path.insert(0,str(REPO/'verification'))
    from colab_inputs import find_archive,download_archive
    credentials,_=google.auth.default()
    service=build('drive','v3',credentials=credentials,cache_discovery=False)
    ARCHIVE=download_archive(service,find_archive(service),WORK/'handoff.zip')
with zipfile.ZipFile(ARCHIVE) as z:
    selected=[i for i in z.infolist() if i.filename.startswith(('data/train-prepared/','data/valid/','data/test/','models/'))]
    for i in selected:
        target=(WORK/i.filename).resolve()
        assert target.is_relative_to(WORK.resolve()), 'unsafe archive path'
        assert (i.external_attr >> 16) & 0o170000 != 0o120000, 'symlink in archive'
        z.extract(i,WORK)
for p in (WORK/'models').glob('*.h5'):
    (REPO/'models').mkdir(exist_ok=True);shutil.copy2(p,REPO/'models'/p.name)
print('입력 압축 해제 완료. 학습 시작 전 코드가 이미지와 모델 해시를 검사합니다.')


In [ ]:
assert 'WORK' in globals() and 'REPO' in globals(), '첫 번째 셀을 먼저 오류 없이 완료하세요.'
VENV=WORK/'venv'
if not VENV.exists():subprocess.run([sys.executable,'-m','venv',str(VENV)],check=True)
PY=str(VENV/'bin/python')
subprocess.run([PY,'-m','pip','install','tensorflow[and-cuda]==2.21.0','keras==3.15.1','numpy==2.3.5','Pillow','pandas','scikit-learn','matplotlib','pyyaml'],check=True)
ENV=os.environ.copy();ENV['PYTHONPATH']=str(REPO/'src')
ENV['TF_FORCE_GPU_ALLOW_GROWTH']='true'
subprocess.run([PY,'-c',"import tensorflow as tf; g=tf.config.list_physical_devices('GPU'); print(g); assert g, 'GPU 미할당: CPU로 자동 진행하지 않습니다.'"],env=ENV,check=True)
DEST=Path('/content/drive/MyDrive/AdversarialAI_GPU_results');DEST.mkdir(exist_ok=True)
print('GPU 확인 완료. 아래에서 모델과 단계 선택 후 한 작업씩 실행하세요.')


In [ ]:
assert 'DEST' in globals() and 'PY' in globals(), '첫 번째와 두 번째 셀을 먼저 완료하세요.'
MODEL='cnn' # cnn 또는 mobilenet
STAGE='train' # train, bim, pgd, jsma
RESUME=False # 중단된 학습 복구: 동일 RUN을 유지하고 True
BATCH=8 # 메모리 절약 설정. 기존 CPU batch32와 다른 새 실험입니다.
RUN='gpu-20261002-cnn-01' # 모델/재시도마다 고유 ID. 평가 때는 학습 RUN을 유지하세요.
assert MODEL in ('cnn','mobilenet') and STAGE in ('train','bim','pgd','jsma')
import re
assert re.fullmatch('[A-Za-z0-9][A-Za-z0-9_-]{0,39}',RUN)
def prepare_output(category,run_id):
    local=REPO/'results/extensions'/category/run_id
    remote=DEST/category/run_id
    if RESUME:
        assert STAGE=='train', '현재 복구 기능은 학습에만 적용됩니다.'
        assert remote.is_dir(), 'Drive에서 이전 학습을 찾지 못했습니다.'
        if not local.exists():shutil.copytree(remote,local,ignore=shutil.ignore_patterns('*.copying','.pending-*'))
        return local,remote
    assert not local.exists() and not local.is_symlink(), '이미 실행한 ID입니다. 새 ID를 쓰세요.'
    assert not remote.exists(), 'Drive에 같은 결과 ID가 있습니다. 덮어쓰지 않습니다.'
    local.parent.mkdir(parents=True,exist_ok=True);remote.parent.mkdir(parents=True,exist_ok=True)
    # Runners reject symlinks: use a real working directory and mirror completed files.
    return local,remote
if STAGE=='train':
    category='adversarial_training';rid=RUN
    args=['--model',MODEL,'--train-dir',str(WORK/'data/train-prepared/images'),'--validation-dir',str(WORK/'data/valid'),'--test-dir',str(WORK/'data/test'),'--epochs','3','--epsilon','0.03','--step-size','0.005','--steps','7','--seed','2026','--batch-size',str(BATCH),'--learning-rate','0.00001','--run-id',rid]
    module='adversarial_ai.training.adversarial_training'
elif STAGE in ('bim','pgd'):
    category='iterative';rid=RUN+'-'+STAGE
    trained=DEST/'adversarial_training'/RUN
    report=json.loads((trained/'training.json').read_text())
    model=trained/'best.keras'
    assert report['status']=='TRAINED_NOT_TEST_EVALUATED', '학습 완료 전 평가 불가'
    digest=hashlib.sha256(model.read_bytes()).hexdigest()
    assert digest==report['trained_model_sha256']
    module='adversarial_ai.evaluation.iterative_evaluation'
    args=['--attack',STAGE,'--steps',str(10 if STAGE=='bim' else 20),'--step-size','0.005','--restarts',str(1 if STAGE=='bim' else 5),'--seed',str(0 if STAGE=='bim' else 2026),'--batch-size',str(BATCH),'--run-id',rid,'--data-dir',str(WORK/'data/test'),'--trained-model',str(model),'--trained-model-sha256',digest,'--trained-model-kind',MODEL]
else:
    category='jsma';rid=RUN+'-jsma'
    module='adversarial_ai.evaluation.jsma_evaluation'
    args=['--model',MODEL,'--defense','none','--theta','1.0','--gamma','0.01','--max-steps',str(246 if MODEL=='cnn' else 753),'--limit','781','--run-id',rid,'--data-dir',str(WORK/'data/test')]
assert not RESUME or STAGE=='train', 'RESUME은 학습 전용입니다.'
if RESUME:args.append('--resume')
local,remote=prepare_output(category,rid)
remote.mkdir(exist_ok=RESUME)
(remote/('resume-'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%f')+'.json' if RESUME else 'launch.json')).write_text(json.dumps({'model':MODEL,'stage':STAGE,'command':[module,*args],'source':COMMIT,'gpu_new_run':True,'resume_requested':RESUME,'resume_policy':'completed_epoch'},indent=2))
# Copy only files unchanged in size/mtime during the transfer; completed writes are mirrored every 30s.
import threading,time
stop=threading.Event()
copied={}
def mirror():
    if not local.exists():return
    # Immutable epoch snapshots are copied before their state.json manifest.
    for p in sorted(local.rglob('*'),key=lambda p:(p.name=='state.json',str(p))):
        if not p.is_file() or any(part.startswith('.pending-') for part in p.relative_to(local).parts) or p.suffix=='.tmp':continue
        dest=remote/p.relative_to(local);dest.parent.mkdir(parents=True,exist_ok=True)
        before=p.stat()
        signature=(before.st_size,before.st_mtime_ns)
        if copied.get(str(p))==signature and dest.is_file():continue
        temp=dest.with_name(dest.name+'.copying')
        try:
            shutil.copy2(p,temp);after=p.stat()
            if signature==(after.st_size,after.st_mtime_ns):
                temp.replace(dest);copied[str(p)]=signature
            else:temp.unlink(missing_ok=True)
        except OSError as e:print('백업 재시도 예정:',type(e).__name__)
def background():
    while not stop.wait(30):mirror()
worker=threading.Thread(target=background,daemon=True);worker.start()
try:
    log=remote/('run-'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%f')+'.log')
    with log.open('w') as stream:
        process=subprocess.Popen([PY,'-u','-m',module,*args],cwd=REPO,env=ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        try:
            for line in process.stdout:
                print(line,end='');stream.write(line);stream.flush()
            if process.wait():raise RuntimeError('실행 오류: Drive의 run 로그를 확인하세요.')
        finally:
            if process.poll() is None:
                process.terminate()
                try:process.wait(timeout=10)
                except subprocess.TimeoutExpired:process.kill();process.wait()
finally:
    stop.set();worker.join();mirror()
if STAGE in ('bim','pgd'):
    subprocess.run([PY,'verification/iterative_result_audit.py',str(local)],cwd=REPO,env=ENV,check=True)
print('단계 실행 종료. 상태 JSON과 CSV 감사를 확인해야 완료로 판단합니다.')


## 해석 및 남은 검증
- 원본 모델의 이전 train/test 중복 노출 가능성은 추가학습으로 제거되지 않습니다.
- 학습용 6147장은 정확한 RGB 중복만 제거했습니다. 근접 중복은 미검사입니다.
- GPU/배치8 결과는 CPU/배치32 재현 완료로 표시하지 않습니다.
- 이 노트북 JSMA 셀은 원본 모델/필터 없음만 실행합니다. Gaussian/Mean 및 모든 추가 실험 완료를 의미하지 않습니다.
- Drive에 복사가 끝난 epoch 체크포인트부터 명시적으로 재개합니다. 미완료 epoch는 반복하며, 중단 없는 실행과 비트 단위 동일 결과는 보장하지 않습니다.
- 같은 코드·데이터·학습 설정만 복구할 수 있습니다. 모델/배치/학습률 등을 바꾸려면 새 RUN을 사용하세요.
- 체크포인트는 epoch마다 전체 모델 두 개를 보관하므로 Drive 공간을 사용합니다.
- 실GPU 통합 실행 미검증. GPU가 없거나 준비 검사가 실패하면 중단합니다.
